# Week 1 · Who gets the spotlight?
Group RAM · 02805 Social Graphs

We ask who receives links, who sends them, and who sits outside the giant component. Run all cells to regenerate the website's interactive figures from the frozen TSV files. An edge A → B means A's article links to B's article within this roster.

In [ ]:
from pathlib import Path
from collections import Counter
import csv
import math
import networkx as nx
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display

ROOT = Path.cwd()
if not (ROOT / "week1_nodes.tsv").exists():
    ROOT = ROOT.parent
assert (ROOT / "week1_nodes.tsv").exists(), "Run from the repository or notebooks folder"
OUT = ROOT / "assets" / "plots"
OUT.mkdir(parents=True, exist_ok=True)
COLORS = ["#ff536d", "#62bcff", "#ffd43b"]
CONFIG = {"responsive": True, "displaylogo": False, "scrollZoom": False}

def publish(fig, name, height=520):
    fig.update_layout(
        template="plotly_dark", paper_bgcolor="#080e20", plot_bgcolor="#080e20",
        font=dict(family="Arial, sans-serif", color="#fff8de", size=14),
        margin=dict(l=60, r=25, t=70, b=65), autosize=True, height=None,
    )
    fig.write_html(OUT / f"{name}.html", include_plotlyjs="directory",
                   config=CONFIG, default_height="100%", div_id=name)
    html_path = OUT / f"{name}.html"
    html = html_path.read_text(encoding="utf-8")
    html = html.replace("<head>", '<head><meta name="viewport" content="width=device-width, initial-scale=1"><style>html,body{margin:0;height:100%;background:#080e20}</style>')
    html_path.write_text(html, encoding="utf-8")
    display(HTML(fig.to_html(include_plotlyjs=True, full_html=False,
                            config=CONFIG, default_height=f"{height}px", div_id=f"notebook-{name}")))


## Load the roster first
Adding all nodes before edges preserves characters with no links in either direction. Validate IDs, endpoints, and duplicates before counting.

In [ ]:
with (ROOT / "week1_nodes.tsv").open(encoding="utf-8") as f:
    roster = list(csv.DictReader((line for line in f if not line.startswith("#")), delimiter="\t"))
with (ROOT / "week1_edges.tsv").open(encoding="utf-8") as f:
    edges = [tuple(line.strip().split("\t")) for line in f if line.strip() and not line.startswith("#")]
names = {row["node_id"]: row["name"] for row in roster}
assert len(names) == len(roster)
assert len(set(edges)) == len(edges)
assert all(a in names and b in names for a, b in edges)
G = nx.DiGraph()
G.add_nodes_from(names)
G.add_edges_from(edges)
incoming, outgoing = dict(G.in_degree()), dict(G.out_degree())
components = sorted(nx.weakly_connected_components(G), key=lambda c: (-len(c), sorted(c)))
isolates = sorted(nx.isolates(G))
assert sum(incoming.values()) == sum(outgoing.values()) == len(edges)
assert (len(G), G.number_of_edges()) == (303, 1784)
assert [len(c) for c in components] == [277, 9] + [1] * 17
print(f"{len(G)} characters · {G.number_of_edges()} links · {len(isolates)} isolates")


## 1. Explore the whole network
The giant component contains 277 characters. A nine-character island and 17 isolates sit separately. Node area grows with in-degree. We omit arrows for readability; hover reveals incoming and outgoing counts. The layout is deterministic and has no geographic meaning.

In [ ]:
positions = {}
for i, comp in enumerate(components[:2]):
    layout = nx.spring_layout(G.subgraph(sorted(comp)).to_undirected(), seed=42, iterations=120)
    for node, xy in layout.items():
        positions[node] = (float(xy[0]) * (1 if i == 0 else .30) + (0 if i == 0 else 1.65),
                           float(xy[1]) * (1 if i == 0 else .30) + (0 if i == 0 else .5))
for i, node in enumerate(isolates):
    positions[node] = (1.3 + (i % 5) * .17, -.35 - (i // 5) * .18)
xs, ys = [], []
for a, b in edges:
    xs.extend([positions[a][0], positions[b][0], None])
    ys.extend([positions[a][1], positions[b][1], None])
network = go.Figure(go.Scatter(x=xs, y=ys, mode="lines",
    line=dict(width=.6, color="rgba(125,169,233,0.20)"), hoverinfo="skip", showlegend=False))
for label, nodes, color in [
    ("Giant · 277", sorted(components[0]), COLORS[0]),
    ("Island · 9", sorted(components[1]), COLORS[1]),
    ("Isolates · 17", isolates, COLORS[2]),
]:
    network.add_trace(go.Scatter(
        x=[positions[n][0] for n in nodes], y=[positions[n][1] for n in nodes],
        mode="markers", name=label, text=[names[n] for n in nodes],
        customdata=[[incoming[n], outgoing[n]] for n in nodes],
        marker=dict(size=[7 + 2 * math.sqrt(incoming[n]) for n in nodes], color=color,
                    line=dict(width=.6, color="#080e20")),
        hovertemplate="<b>%{text}</b><br>Incoming: %{customdata[0]}<br>Outgoing: %{customdata[1]}<extra></extra>"
    ))
network.update_layout(title="A universe with an outside", dragmode="pan",
    legend=dict(orientation="h", x=0, y=1.08),
    xaxis=dict(visible=False), yaxis=dict(visible=False, scaleanchor="x"))
publish(network, "network", 650)
print("Nine-character island:", ", ".join(names[n] for n in sorted(components[1])))
print("Isolates:", ", ".join(names[n] for n in isolates))


## 2. Incoming versus outgoing
Each dot is a character. The diagonal marks equal incoming and outgoing counts. Spider-Man leads incoming links (106); Betsy Braddock leads outgoing links (28). These measure Wikipedia article structure, not friendship or audience popularity.

In [ ]:
nodes = sorted(G)
comparison = go.Figure(go.Scatter(
    x=[outgoing[n] for n in nodes], y=[incoming[n] for n in nodes],
    mode="markers", text=[names[n] for n in nodes],
    marker=dict(size=9, color=COLORS[1], opacity=.7),
    hovertemplate="<b>%{text}</b><br>Outgoing: %{x}<br>Incoming: %{y}<extra></extra>",
    name="Characters"))
comparison.add_trace(go.Scatter(x=[0, 28], y=[0, 28], mode="lines",
    line=dict(color=COLORS[2], dash="dot"), name="Equal in and out", hoverinfo="skip"))
comparison.update_layout(title="Incoming vs. outgoing",
    xaxis_title="Out-degree · links sent", yaxis_title="In-degree · links received",
    legend=dict(orientation="h", y=1.1))
publish(comparison, "comparison")


## 3. Degree distributions
Count characters at each exact degree. Switch axes with the buttons. Log–log omits zero-degree observations because log(0) is undefined; the linear view retains them. A long tail alone does not establish a power law.

In [ ]:
distribution = go.Figure()
for degree, label, color, symbol in [
    (incoming, "Incoming", COLORS[0], "circle"),
    (outgoing, "Outgoing", COLORS[1], "diamond"),
]:
    counts = sorted(Counter(degree.values()).items())
    distribution.add_trace(go.Scatter(x=[k for k, count in counts], y=[count for k, count in counts],
        mode="markers", name=label, marker=dict(color=color, size=9, symbol=symbol),
        hovertemplate="Degree %{x}: %{y} characters<extra>%{fullData.name}</extra>"))
distribution.update_layout(title="Degree distributions",
    xaxis_title="Degree · links per character", yaxis_title="Number of characters",
    legend=dict(orientation="h", y=1.12),
    updatemenus=[dict(type="buttons", direction="right", x=1, xanchor="right", y=1.18,
        bgcolor="#172443", font=dict(color="#fff8de"),
        buttons=[dict(label="Linear", method="relayout", args=[{"xaxis.type":"linear","yaxis.type":"linear","xaxis.autorange":True,"yaxis.autorange":True}]),
                 dict(label="Log–log", method="relayout", args=[{"xaxis.type":"log","yaxis.type":"log","xaxis.autorange":True,"yaxis.autorange":True}])])])
publish(distribution, "distribution")
print(f"Zero in-degree: {sum(v == 0 for v in incoming.values())}; zero out-degree: {sum(v == 0 for v in outgoing.values())}")


## 4. The two leaderboards
A recurring character may collect references across many pages, while a page covering many relationships may link out widely. This is an interpretation; the counts cannot establish the editorial reasons without reading the articles.

In [ ]:
rankings = make_subplots(rows=2, cols=1, subplot_titles=("Most linked to", "Most links out"), vertical_spacing=.22)
for col, degree, color in [(1, incoming, COLORS[0]), (2, outgoing, COLORS[1])]:
    top = sorted(degree, key=lambda n: (-degree[n], names[n]))[:5][::-1]
    rankings.add_trace(go.Bar(y=[names[n].split(" (")[0].replace("Cloak and Dagger", "Cloak & Dagger") for n in top], x=[degree[n] for n in top],
        orientation="h", marker_color=color, text=[degree[n] for n in top],
        textposition="auto", showlegend=False, hovertemplate="%{y}: %{x} links<extra></extra>"), row=col, col=1)
    rankings.update_xaxes(title_text="Links", row=col, col=1)
rankings.update_layout(title="The degree leaders")
rankings.update_yaxes(automargin=True)
publish(rankings, "rankings", 650)


## Source and limits
Frozen week-one release dated 2026-08-26 in the supplied file headers: [course data page](https://sunelehmann.com/socialgraphs2026-web/data/). We have not re-crawled Wikipedia. Components are weakly connected (edge direction ignored). An isolate has no links within this roster, not necessarily across Wikipedia.

The website embeds the HTML figures exported by this notebook. Edit the cells and run all, or push to main to have GitHub Actions execute and publish them.